# Lab 15: Chunking and embeddings

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/prekshapraveen/Documents/lantern-lab2-submission.nosync | LLM mode: replay


In [2]:
recs = sd.records("deal_a") + sd.records("deal_b")

In [3]:
# From the slide: Block-aware chunking on top of your CS1 records
ntok = lambda s: max(1, len(s) // 4)  # ~chars/4

def block_chunks(blocks, budget=400):
    """blocks: CS1 records in reading order."""
    out, cur, sec = [], [], None
    def flush():
        if cur:
            out.append({"section": sec,
                "block_ids": [b["block_id"] for b in cur],
                "text": "\n".join(b["text"] for b in cur)})
    for b in blocks:
        if b["block_type"] == "Table":   # own chunk
            flush(); cur = []
            out.append({"section": b["section"],
                        "block_ids": [b["block_id"]],
                        "text": b["text"],
                        "is_table": True})
            continue
        size = sum(ntok(x["text"]) for x in cur)
        if cur and (b["section"] != sec or
                    size + ntok(b["text"]) > budget):
            flush(); cur = []
        cur.append(b); sec = b["section"]
    flush()
    return out

In [4]:
for c in block_chunks(sd.records("deal_a"), budget=120)[:4]: print(c["section"], c["block_ids"], ntok(c["text"]))
section_chunks = {}
for r in sd.records("deal_a"): section_chunks.setdefault(r["section"], []).append(r["text"])
print(len(section_chunks), "section chunks")
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    base = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40).split_text(sd.cim_text())
    print(len(base), "recursive-character chunks (the baseline)")
except ImportError:
    print("pip install langchain-text-splitters for the baseline")

Company Overview ['p0001_b001', 'p0001_b002'] 82
Investment Highlights ['p0002_b001', 'p0002_b002', 'p0002_b003'] 56
Customers and Retention ['p0003_b001', 'p0003_b002'] 48
Financial Summary ['p0004_b001', 'p0004_b002'] 69
8 section chunks
5 recursive-character chunks (the baseline)


Embeddings go through `llm.embed` (part 4), cached by text hash. In class we use `lsa-sample-v1`, an offline TF-IDF + SVD model seeded into the cache, so this runs with no download. Case Study 2 requires one hosted and one open model: with `sentence-transformers` installed, switch to record mode and embed with your open model.

In [5]:
import numpy as np
from src import sample_lsa
texts = [r["text"] for r in recs]
V = llm.embed(texts, sample_lsa.MODEL, compute=sample_lsa.encode)
q = llm.embed(["customer concentration"], sample_lsa.MODEL, compute=sample_lsa.encode)[0]
for i in np.argsort(-(V @ q))[:3]: print(round(float(V[i] @ q), 3), recs[i]["doc_id"], recs[i]["block_id"], recs[i]["text"][:70])

0.841 DR-deal_a-001 p0003_b001 Wren's customer base is highly diversified: no single customer represe
0.646 DR-deal_a-002 t002 Q: How concentrated is the customer base? A (CEO): Our largest custome
0.283 DR-deal_a-003 p0001_b003 Section 14.2 Change of Control: Customer may terminate on 60 days noti


**Exercise 15.** Chunk with budgets of 60 and 400 tokens. How many chunks does each give, and which would you expect to retrieve the NRR answer better?

In [6]:
for budget in (60, 400):
    ch = block_chunks(sd.records("deal_a"), budget=budget)
    hits = [(c["section"], c["block_ids"], ntok(c["text"]))
            for c in ch if "retention" in c["text"].lower()]
    print(budget, len(ch), "chunks"); print(*hits, sep="\n")

60 13 chunks
('Customers and Retention', ['p0003_b001', 'p0003_b002'], 48)
('Management call', ['t003'], 42)
400 8 chunks
('Customers and Retention', ['p0003_b001', 'p0003_b002'], 48)
('Management call', ['t001', 't002', 't003', 't004'], 126)


### Exercise 15: answer

**How many chunks?** With a 60-token budget, block-aware chunking gives **13 chunks**; with 400 it
gives **8**. At 400 every section fits in one chunk (no Wren section is longer than 400 tokens),
so the section boundaries, not the budget, decide the cuts.

**Which retrieves the NRR answer better?** The **60-token** budget. The answer to "What is net
revenue retention?" is the CFO's reply in transcript block `t003` (about 109% on a trailing
twelve-month basis, versus the CIM's 118%, which is the best quarter annualized).
- At 60, `t003` is its own chunk of 42 tokens, so its embedding is about NRR and little else.
- At 400, `t003` is merged with `t001`, `t002` and `t004` into one 126-token chunk that also covers
  growth drivers, customer concentration and change-of-control contracts, which dilutes the match.

The "Customers and Retention" CIM chunk (`p0003_b001`, `p0003_b002`, 48 tokens) is the same at both
budgets, so it doesn't change the comparison.

**Takeaway:** smaller chunks raise precision for a focused question; bigger chunks keep more context.
Which matters more depends on the question type, so it should be measured, not assumed.